# AML - SAIGE-GENE SET-BASED ANALYSIS - FIL NULL MODEL 
The following script is used to fit the null model for the set-based SAIGE-GENE analysis. The options useSparseGRMtoFitNULL, useSparseGRMforVarRatio, and isCateVarianceRatio must be set to TRUE. This script uses the sparse GRM, 2,000 randomly selected variants in the specified MAC categories, and the phenotype file with covariates to fit the null model.

In [1]:
# Load packages 
import os

In [2]:
# Set environment variable for workspace bucket 
my_bucket = os.getenv('WORKSPACE_BUCKET')
# Set environment variable for workspace bucket 
my_bucket

'gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe'

# COMMAND 

In [3]:
%%writefile saige_gene_step01.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    for pheno in aml_biosample_lower_50th_percentile aml_biosample_upper_50th_percentile
    do
    echo "Processing phenotype: ${pheno}"
    
    step1_fitNULLGLMM.R \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --plinkFile="${variance_files}/${pop}_array.forCate_vr" \
        --phenoFile="${phenotype}/${pheno}_files/${pop}_phenotype_file.txt" \
        --useSparseGRMtoFitNULL=TRUE \
        --useSparseGRMforVarRatio=TRUE \
        --IsOverwriteVarianceRatioFile=TRUE \
        --isCateVarianceRatio=TRUE \
        --phenoCol=aml_phenotype \
        --traitType=binary \
        --covarColList=sex,age,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10 \
        --qCovarColList=sex \
        --sampleIDColinphenoFile=person_id \
        --outputPrefix="${output}/${pop}_${pheno}_null_glmm" 
    done
done


Overwriting saige_gene_step01.sh


In [4]:
# Save script to workspace bucket 
!gsutil cp saige_gene_step01.sh {my_bucket}/data/scripts/

Copying file://saige_gene_step01.sh [Content-Type=text/x-sh]...
/ [1 files][  1.2 KiB/  1.2 KiB]                                                
Operation completed over 1 objects/1.2 KiB.                                      


In [5]:
# Check script in workspace bucket 
!gsutil cat {my_bucket}/data/scripts/saige_gene_step01.sh 

#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    for pheno in aml_biosample_lower_50th_percentile aml_biosample_upper_50th_percentile
    do
    echo "Processing phenotype: ${pheno}"
    
    step1_fitNULLGLMM.R \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --plinkFile="${variance_files}/${pop}_array.forCate_vr" \
        --phenoFile="${phenotype}/${pheno}_files/${pop}_phenotype_file.txt" \
        --useSparseGRMtoFitNULL=TRUE \
        --useSparseGRMforVarRatio=TRUE \
        --IsOverwriteVarianceRatioFile=TRUE \
        --isCateVarianceRatio=TRUE \
        --pheno

# SUBMIT COMMAND 

In [6]:
%%bash --out saige_gene_step01

source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 50 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input-recursive phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files/" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --env phenotype="${WORKSPACE_BUCKET}/data/saige_phenotype_files" \
  --env grm="${WORKSPACE_BUCKET}/data/saige_step00/grm" \
  --env variance_files="${WORKSPACE_BUCKET}/data/saige_step01/variance_files" \
  --env output="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm" \
  --script "${WORKSPACE_BUCKET}/data/scripts/saige_gene_step01.sh"


Job properties:
  job-id: saige-gene--vienose--260416-100333-29
  job-name: saige-gene-step01
  user-id: vienose
Provider internal-id (operation): projects/terra-vpc-sc-fc4465fd/locations/us-central1/jobs/saige-gene--vienose--260416-100333-29-0-0
Launched job-id: saige-gene--vienose--260416-100333-29
To check the status, run:
  dstat --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'saige-gene--vienose--260416-100333-29' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'saige-gene--vienose--260416-100333-29' --users 'vienose'


In [9]:
!  dstat --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'saige-gene--vienose--260416-100333-29' --users 'vienose' --status '*'

Job Name         Status    Last Update
---------------  --------  --------------
saige-gene-s...  SUCCESS   04-16 10:12:33

